In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np
import joblib
import os

In [ ]:
# =========================== Eliminarlo e Importarlo desde el .py =========================== #

# Métricas de evaluación utilizadas en el paper original

def paper_r2(y_true, y_pred):
    """Paper's metric: Pearson correlation"""
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return corr 

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [ ]:
# Definition of attributes and targets

# =========================== 21 atributos seleccionados =========================== #

selected_21_attributes = [
    'np_without_modification', 'surface_modification', 'size_tem', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'centrifugation_speed', 'centrifugation_time', 'centrifugation_temperature', 'np_type',
    'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 'incubation_culture',
    'incubation_time', 'incubation_temperature', 'centrifugation_repetitions', 'modification_type'
]


categorical_21_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 10 atributos seleccionados =========================== #

selected_10_attributes = [
    'np_without_modification', 'surface_modification', 
    'incubation_protein_source', 'size_tem', 'zeta_potential', 'incubation_plasma_concentration', 
    'incubation_np_concentration','centrifugation_speed', 
    'centrifugation_time', 'centrifugation_temperature'    
]

categorical_10_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
]

# =========================== 17 atributos seleccionados (Jair) =========================== #

selected_17_attributes = [
    'np_without_modification', 'surface_modification', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'np_type', 'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 
    'incubation_culture', 'incubation_time', 'incubation_temperature', 'modification_type'
]

categorical_17_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 16 atributos seleccionados (sin np_type para el entrenamiento) =========================== #

selected_16_attributes = [
    'np_without_modification', 'surface_modification', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration', 
    'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 
    'incubation_culture', 'incubation_time', 'incubation_temperature', 'modification_type'
]

categorical_16_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 178 Proteinas =========================== #

protein_targets = [
    "P01871", "P01024", "P02647", "P02649", "P02768", "P04004", "P01834", 
    "P10909", "P02652", "P00734", "P01009", "P01042", "P01857", "P01859", 
    "P01023", "P01619", "P04196", "P02656", "P04114", "P02787", "P02766", 
    "P06396", "P06727", "P0C0L5", "P02671", "P08603", "P02765", "Q14624", 
    "P0DOY2", "P01008", "P68871", "P01764", "P04003", "P0C0L4", "P01876", 
    "P02749", "P02675", "P01860", "P01011", "P00747", "P02774", "P00751", 
    "P19823", "P08697", "P19827", "P69905", "P02760", "P02748", "P02679", 
    "P02790", "B9A064", "P07996", "P05155", "P01766", "P12259", "P00738",
    "P02751", "P02654", "P04406", "P00739", "P02655", "P00736", "P07225", 
    "P05154", "P09871", "P60709", "P03952", "P02747", "P05546", "P02746",
    "P27169", "P01019", "P35542", "P04217", "Q14520", "P02743", "P02763", 
    "P49908", "O43866", "P18428", "P55056", "P04264", "P00748", "P01591", 
    "P01861", "Q92954", "P02776", "P0DJI8", "P01031", "P05156", "P13671", 
    "Q03591", "P06312", "P00740", "P01615", "P00742", "P04070", "O14791", 
    "P05090", "P02745", "P00488", "Q13103", "P01877", "P01700", "P10643", 
    "Q9UK55", "P03951", "Q06033", "Q96IY4", "P13645", "P04433", "P07358",
    "P27918", "P05452", "P20851", "P07357", "P07360", "P01599", "O95445", 
    "Q9BXR6", "P02775", "P02741", "P35579", "P15169", "Q13790", "P35858", 
    "P49747", "P36955", "P19652", "P07737", "P22891", "P35908", "P80748", 
    "P08514", "P01593", "Q04756", "P06733", "P23528", "P63104", "P18065", 
    "P08519", "Q86UX7", "P02753", "Q5TB80", "P22352", "P00746", "P35443", 
    "P62937", "P10720", "P48740", "P25311", "P43652", "P35527", "Q9Y490", 
    "P05106", "P17936", "P18206", "P02788", "P06702", "P01701", "Q6Q788", 
    "Q96PD5", "Q9UGM5", "O00391", "P11142", "P14618", "P23142", "P68366",
    "P12814", "P61224", "Q13201", "P67936", "P0DOY3", "P08709", "P01034", 
    "P81605", "P02533", "P11226"]

In [ ]:
def fit_csv(csv_filename, selected_attributes, categorical_attributes, protein_targets, dictionary_params=None, directory='./modelos_guardados/', show=False):
    """
    Carga el dataset, preprocesa los datos, entrena un RandomForestRegressor 
    para cada proteína objetivo y guarda los modelos y las columnas esperadas.
    """
    if not all([selected_attributes, categorical_attributes, protein_targets]):
        print("Error: Faltan especificar atributos, variables categóricas o targets.")
        return

    # Cargamos el dataset o el archivo .csv
    df = pd.read_csv(f"{csv_filename}.csv")
    # Seleccionamos los atributos de interes del dataset
    X = df[selected_attributes]
    # Hacemos la codificación One-hot a los atributos categóricos
    X = pd.get_dummies(X, columns=categorical_attributes)

    # Definimos en qué directorio se van a almacenar los modelos
    os.makedirs(directory, exist_ok=True)
    if show: print(f"Directorio listo: {directory}")

    # Almacenamos la estructura de columnas para las predicciones
    columns_filepath = os.path.join(directory, 'training_columns.joblib')
    joblib.dump(X.columns, columns_filepath)
    if show: print(f"Columnas de entrenamiento guardadas en: {columns_filepath}\n")

    # Entrenamiento y Serialización de todos los Modelos
    for target in protein_targets:
        y = df[target].values

        # Verificamos si pasamos un diccionario de parametros óptimos y si la proteina analizada existe
        if diccionario_params is not None and target in diccionario_params:
            # Extraemos los parámetros de esta proteína específica
            params_optimos = diccionario_params[target]
            
            # Desempaquetamos el diccionario en el modelo (con **)
            model = RandomForestRegressor(random_state=42, **params_optimos)
            if show: print(f"Entrenando {target} con parámetros óptimos...")
            
        else:
            # Si no hay diccionario o no se encuentra la proteina, usamos los siguientes parametros
            model = RandomForestRegressor(n_estimators=500, random_state=42, max_samples=0.3)
            if show: print(f"Entrenando {target} con parámetros por defecto...")

        # Entrenamos el modelo de Random Forest
        model.fit(X, y)

        # Guardamos el modelo RF de una proteina
        filename = f'random_forest_regressor_model_{target}.joblib'
        filepath = os.path.join(directory, filename)
        joblib.dump(model, filepath)
        
        if show:
            print(f"Modelo guardado: {filename}")
            
    print("\n========================================================================")
    print(f"Éxito: Se han guardado los modelos y metadatos en: {directory}")

In [ ]:
# Entrenamiento del modelo que fue entrenado con los NP de tipo Liposomas sin el atributo 'np_type'

# Estructurar bien la llamada a la función de entrenamiento o ajuste.
# Hacer un diccionario para que en base al modelo utilizado o al csv a predecir que le asigne un nombre u otro(?

csv_filename_train='liposomes_dataset-np_type'
output_filename='./liposome_models-np_type'

# Definimos la ruta en donde se encuentra el dataset
input_filepath = os.path.join('..', 'datasets', csv_filename)

# Definimos la ruta en donde se guardaran las salidas
output_dir = 'models'
output_filepath = os.path.join(output_dir, output_filename)
os.makedirs(output_filepath, exist_ok=True)

# Definimos la ruta en donde se encuentra el diccionario de los parametros optimos en caso que los hubiera
filename='optimal_parameters'
dictionary_filename = joblib.load('hiperparametros_optimos.joblib')
dictionary_filepath=os.path.join("..", filename, dictionary_filename)


fit_csv(
    csv_filename=input_filepath, 
    selected_attributes=selected_16_attributes, 
    categorical_attributes=categorical_16_attributes,
    protein_targets=protein_targets,
    # dictionary_params=dictionary_filepath,   # Descomentar en caso de utilizar los Parametros optimos
    directory=outputh_filepath
)

In [ ]:
# Cambiar el nombre para indicar de que son los Hiperparametros Optimos del Dataset de Liposomas sin el atributo np_type
params_opt = joblib.load('hiperparametros_optimos.joblib')
csv_filename_train="./datasets/liposomes_dataset-np_type" # Esta guardado en esa carpeta
directory='./liposome_models_opt-np_type'

# csv_filename_train='liposomes_dataset-np_type'

fit_csv(
    csv_filename=csv_filename_train, 
    selected_attributes=selected_16_attributes, 
    categorical_attributes=categorical_16_attributes, 
    protein_targets=protein_targets,
    diccionario_params=params_opt,
    directory=directory
)